# 05 - Bonus: Association Rules & Clustering
Run association rules mining and clustering analysis.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

base_dir = Path('..').resolve()
processed = pd.read_csv(base_dir / 'data' / 'processed' / 'processed_full.csv')
features = processed.drop(columns=['defects'])

# Association Rules Mining
discretizer = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='quantile')
binned = pd.DataFrame(discretizer.fit_transform(features), columns=features.columns)
binned = binned.applymap(lambda x: ['Low', 'Medium', 'High'][int(x)])
binned['defects'] = processed['defects'].map({0: 'Low', 1: 'High'})

one_hot = pd.get_dummies(binned)
frequent = apriori(one_hot, min_support=0.1, use_colnames=True)
rules = association_rules(frequent, metric='confidence', min_threshold=0.6)
rules = rules[rules['lift'] > 1.2]
rules = rules[rules['consequents'].astype(str).str.contains('defects_High')]
rules = rules.sort_values(by='lift', ascending=False).head(10)
rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]

# Clustering - KMeans
inertia = []
for k in range(2, 10):
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(features)
    inertia.append(kmeans.inertia_)

plt.figure()
plt.plot(range(2, 10), inertia, marker='o')
plt.title('Elbow Method')
plt.xlabel('k')
plt.ylabel('Inertia')
plt.show()

optimal_k = 3
kmeans = KMeans(n_clusters=optimal_k, random_state=42)
clusters = kmeans.fit_predict(features)

pca = PCA(n_components=2, random_state=42)
pca_data = pca.fit_transform(features)
plt.figure()
sns.scatterplot(x=pca_data[:, 0], y=pca_data[:, 1], hue=clusters, palette='tab10')
plt.title('KMeans Clusters (PCA)')
plt.show()

# DBSCAN
dbscan = DBSCAN(eps=0.5, min_samples=5)
db_labels = dbscan.fit_predict(features)
plt.figure()
sns.scatterplot(x=pca_data[:, 0], y=pca_data[:, 1], hue=db_labels, palette='tab10')
plt.title('DBSCAN Clusters (PCA)')
plt.show()

# Hierarchical Clustering
agg = AgglomerativeClustering(n_clusters=3)
agg_labels = agg.fit_predict(features)
plt.figure()
sns.scatterplot(x=pca_data[:, 0], y=pca_data[:, 1], hue=agg_labels, palette='tab10')
plt.title('Hierarchical Clusters (PCA)')
plt.show()

cluster_profile = processed.copy()
cluster_profile['cluster'] = clusters
summary = cluster_profile.groupby('cluster').agg(defect_rate=('defects', 'mean'))
summary
